In [ ]:
import pandas as pd
from typing import List
from chronos import Chronos2Pipeline
import numpy as np
DATA_DIR = "../../seeds/data/raw/aws_clean_baseline.parquet"
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2")
df = pd.read_parquet(DATA_DIR)
df.tail(20)

In [ ]:
df.iloc[0]

In [ ]:
df.temp_c.values[-20:len(df.temp_c.values)-1]

In [ ]:
df.temp_c.values[-1]

In [ ]:
N=20
inputs = np.array(
        [
            [
                df.temp_c.values[-N:len(df.temp_c.values)-1],
                df.humidity_pct.values[-N:len(df.humidity_pct)-1],
                df.pressure_hpa.values[-N:len(df.pressure_hpa)-1],
            ]
        ]
    )
inputs

In [ ]:
quantiles, mean = pipeline.predict_quantiles(
        inputs, prediction_length=1, quantile_levels=[0.05, 0.5, 0.95]
    )

In [ ]:
forecast = quantiles[0].tolist()

In [ ]:
actual = [12,df.humidity_pct.values[-1],1000]
actual

In [ ]:
for i,reading in enumerate(forecast):
    p5,p50,p95 = reading[0]
    corridor_width = p95 - p5
    upper_breach = np.maximum(0.0, actual[i] - p95)
    lower_breach = np.maximum(0.0, p5 - actual[i])
    total_breach = upper_breach + lower_breach
    severity = total_breach / corridor_width
    severity = np.round(severity, 4)
    print(severity)
    predicted_anomly = severity > 0.8
    print(predicted_anomly)
    

# Evaluation

In [15]:
import pandas as pd
from typing import List
from chronos import Chronos2Pipeline
import numpy as np
DATA_DIR = "../../seeds/aws_evaluation_dataset.parquet"
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2")
df = pd.read_parquet(DATA_DIR)
df.tail(20)

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 5466.78it/s]


,timestamp,station_id,name,lat,lon,elevation_m,temp_c,humidity_pct,pressure_hpa,is_anomaly,anomaly_type,affected_sensor
263140,2024-12-31 04:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,14.1,93.0,1010.2,0,normal,none
263141,2024-12-31 05:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,14.1,92.0,1010.9,0,normal,none
263142,2024-12-31 06:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,13.7,98.0,1012.1,0,normal,none
263143,2024-12-31 07:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,15.6,94.0,1013.0,0,normal,none
263144,2024-12-31 08:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,17.0,89.0,1013.8,0,normal,none
263145,2024-12-31 09:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,18.3,86.0,1014.2,0,normal,none
263146,2024-12-31 10:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,20.2,79.0,1013.0,0,normal,none
263147,2024-12-31 11:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,22.1,69.0,1011.3,0,normal,none
263148,2024-12-31 12:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,23.2,62.0,1009.9,0,normal,none
263149,2024-12-31 13:00:00,GAU001,Guwahati (Borjhar),26.106,91.585,54.0,23.7,59.0,1008.6,0,normal,none


In [16]:
from predictor import predict

In [17]:
correct_predictions = 0
total_predictions = 0

N = 20
# Ensure we don't loop past the end of the dataframe
max_iterations = min(10**5, len(df) - N) 

for i in range(max_iterations):
    # 1. Grab the rolling window of size N
    window_df = df.iloc[i : i + N]
    
    # 2. Make the prediction
    result = predict(pipeline, window_df)
    pred_is_anomaly = bool(result["is_anomaly"])
    
    # 3. Get the actual label 
    # Assuming the 'actual' label is the latest/last row in this N-sized window
    actual_is_anomaly = bool(df.iloc[i + N - 1]["is_anomaly"])
    
    # 4. Calculate real Accuracy (Correct matches / Total predictions)
    if pred_is_anomaly == actual_is_anomaly:
        correct_predictions += 1
        
    total_predictions += 1
    
    # 5. Print results (formatted to 2 decimal places)
    accuracy = (correct_predictions / total_predictions) * 100
    if i % 100 == 0:
        print(f"[{i}] Window Accuracy: {accuracy:.2f}%")

[0] Window Accuracy: 100.00%
[100] Window Accuracy: 96.04%
[200] Window Accuracy: 94.53%
[300] Window Accuracy: 96.35%
[400] Window Accuracy: 97.26%
[500] Window Accuracy: 97.41%
[600] Window Accuracy: 97.50%


KeyboardInterrupt: 